# StormEngine V8 — ConvGRU local parameter tuning

ConvGRU has won both Processor-family seeds. This stage fixes the selected 96-channel latent interface and changes only ConvGRU depth and spatial kernel. The existing converged 2-layer/3x3 seed-42 run is reused; it is not retrained.

New converged candidates: 1 layer/3x3, 3 layers/3x3, and 2 layers/5x5. Training remains ERA5 2013–2015, validation remains 2016, stride remains 3 hours, and 2017 remains unread.

In [1]:
from pathlib import Path
import json, subprocess, sys, torch
here = Path.cwd().resolve()
REPO = here if (here / 'pyproject.toml').is_file() else here.parent
assert (REPO / 'pyproject.toml').is_file(), REPO
BASELINE = REPO / 'artifacts' / 'v8_processor_dev3y_convgru_seed42'
assert (BASELINE / 'develop_summary.json').is_file(), 'The converged L2/K3 seed-42 baseline is required.'
CONFIGS = {
    'L1-K3': REPO / 'configs' / 'v8_processor_dev3y_convgru_l1k3.yaml',
    'L3-K3': REPO / 'configs' / 'v8_processor_dev3y_convgru_l3k3.yaml',
    'L2-K5': REPO / 'configs' / 'v8_processor_dev3y_convgru_l2k5.yaml',
}
OUTPUTS = {
    'L1-K3': REPO / 'artifacts' / 'v8_processor_dev3y_convgru_l1k3_seed42',
    'L3-K3': REPO / 'artifacts' / 'v8_processor_dev3y_convgru_l3k3_seed42',
    'L2-K5': REPO / 'artifacts' / 'v8_processor_dev3y_convgru_l2k5_seed42',
}
print('Repository:', REPO)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CUDA unavailable')

Repository: D:\Documents\py_projects\StormEngine-DL\StormEngine-DL
GPU: NVIDIA GeForce RTX 4060 Laptop GPU


## 1. Contract and combined-memory preflight

All three candidates perform a real forward/backward pass. The combined check includes conservative CUDA-process overhead. Passing memory does not mean that three jobs will be faster together; they still share one GPU.

In [2]:
def preflight(config):
    command = [sys.executable, '-u', str(REPO / 'scripts' / 'train_dense_processor.py'),
               'preflight', '--device', 'cuda', '--config', str(config)]
    completed = subprocess.run(command, cwd=REPO, check=True, text=True, capture_output=True)
    text = completed.stdout
    result = json.loads(text[text.index('{'):])
    print(result['contract']['processor'],
          'parameters=', f"{result['contract']['trainable_parameters']:,}",
          'peak=', f"{result['peak_cuda_allocated_gib']:.2f} GiB")
    return result
profiles = {name: preflight(config) for name, config in CONFIGS.items()}
free_bytes, total_bytes = torch.cuda.mem_get_info()
estimated = 1.35 * sum(item['peak_cuda_allocated_bytes'] for item in profiles.values()) + 1.75 * 1024**3
PARALLEL_MEMORY_SAFE = estimated < free_bytes
print(f'GPU free/total: {free_bytes/1024**3:.2f}/{total_bytes/1024**3:.2f} GiB')
print(f'Conservative three-process requirement: {estimated/1024**3:.2f} GiB')
print('Three-process memory check:', 'PASS' if PARALLEL_MEMORY_SAFE else 'FAIL')

{'family': 'convgru', 'latent_channels': 96, 'layers': 1, 'kernel_size': 3} parameters= 499,013 peak= 0.57 GiB
{'family': 'convgru', 'latent_channels': 96, 'layers': 3, 'kernel_size': 3} parameters= 1,494,917 peak= 1.52 GiB
{'family': 'convgru', 'latent_channels': 96, 'layers': 2, 'kernel_size': 5} parameters= 2,766,437 peak= 1.05 GiB
GPU free/total: 6.93/8.00 GiB
Conservative three-process requirement: 5.98 GiB
Three-process memory check: PASS


## 2. Isolated smoke checks

In [4]:
RUN_SMOKES = True
if RUN_SMOKES:
    for name, config in CONFIGS.items():
        smoke = OUTPUTS[name].with_name(OUTPUTS[name].name + '_smoke')
        if (smoke / 'smoke_summary.json').is_file():
            print('Already complete:', smoke)
            continue
        subprocess.run([sys.executable, '-u', str(REPO / 'scripts' / 'train_dense_processor.py'),
                        'smoke', '--device', 'cuda', '--config', str(config),
                        '--output-dir', str(smoke)], cwd=REPO, check=True)
else:
    print('Set RUN_SMOKES=True and run once before the full jobs.')

Already complete: D:\Documents\py_projects\StormEngine-DL\StormEngine-DL\artifacts\v8_processor_dev3y_convgru_l1k3_seed42_smoke


## 3. Launch the three converged candidates

`MAX_PARALLEL=2` is recommended even if all three fit in memory, because recurrent convolutions compete heavily for GPU compute. Set it to 3 only for unattended scheduling after the memory check passes. Each candidate resumes its own `last.pt` automatically.

In [5]:
RUN_TUNING = True
MAX_PARALLEL = 2
if RUN_TUNING:
    assert sys.platform == 'win32', 'Launcher is intended for the Windows CUDA computer.'
    assert 1 <= MAX_PARALLEL <= 3
    if MAX_PARALLEL == 3:
        assert PARALLEL_MEMORY_SAFE, 'Combined peak-memory check failed.'
    pending = []
    for name, config in CONFIGS.items():
        if (OUTPUTS[name] / 'develop_summary.json').is_file():
            print('Already complete:', name)
            continue
        command = [sys.executable, '-u', str(REPO / 'scripts' / 'train_dense_processor.py'),
                   'develop', '--device', 'cuda', '--config', str(config)]
        checkpoint = OUTPUTS[name] / 'last.pt'
        if checkpoint.is_file():
            command += ['--resume', str(checkpoint)]
        pending.append((name, command))
    active = []
    while pending or active:
        while pending and len(active) < MAX_PARALLEL:
            name, command = pending.pop(0)
            process = subprocess.Popen(command, cwd=REPO,
                                       creationflags=subprocess.CREATE_NEW_CONSOLE)
            active.append((name, process))
            print('Launched:', name, process.pid)
        name, process = active.pop(0)
        code = process.wait()
        if code != 0:
            raise RuntimeError(f'{name} exited with code {code}')
        print('Completed:', name)
else:
    print('Set RUN_TUNING=True after preflight and smoke checks.')

Launched: L1-K3 16480
Launched: L3-K3 37656
Completed: L1-K3
Launched: L2-K5 2768
Completed: L3-K3
Completed: L2-K5


## 4. Local tuning comparison

The comparison decides whether a depth-by-kernel interaction run is needed. Do not start seed-43 replication until this adaptive decision is made.

In [ ]:
summaries = [BASELINE / 'develop_summary.json',
             *(OUTPUTS[name] / 'develop_summary.json' for name in OUTPUTS)]
if all(path.is_file() for path in summaries):
    subprocess.run([sys.executable, '-u', str(REPO / 'scripts' / 'compare_convgru_local_tuning.py'),
                    *map(str, summaries), '--output',
                    str(REPO / 'artifacts' / 'v8_processor_convgru_local_tuning_seed42.json')],
                   cwd=REPO, check=True)
else:
    print('Wait until all three new develop_summary.json files exist.')